In [1]:
# MIF 2026 — auditoria reprodutível do portfólio de canais
# Usa os artefatos anônimos canônicos por padrão. Para uma validação isolada,
# defina MIF_PORTFOLIO_AUDIT_INPUT_DIR para outro diretório contendo os dois JSONs.
import json
import os
from decimal import Decimal
from pathlib import Path

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / '_codex').is_dir():
            return candidate
    raise FileNotFoundError('Não foi possível localizar a raiz do repositório.')

def show_rows(rows, columns):
    widths = {column: max(len(column), *(len(str(row.get(column, ''))) for row in rows)) for column in columns}
    line = ' | '.join('-' * widths[column] for column in columns)
    print(' | '.join(column.ljust(widths[column]) for column in columns))
    print(line)
    for row in rows:
        print(' | '.join(str(row.get(column, '')).ljust(widths[column]) for column in columns))

repo_root = find_repo_root(Path.cwd().resolve())
canonical_input_dir = repo_root / '_codex/analyses/mif_2026_channels/modular_dist/portfolio'
input_dir = Path(os.environ.get('MIF_PORTFOLIO_AUDIT_INPUT_DIR', canonical_input_dir)).expanduser().resolve()
summary_path = input_dir / 'summary.json'
simulator_path = input_dir / 'simulator.json'
if not summary_path.is_file() or not simulator_path.is_file():
    raise FileNotFoundError(f'Esperados summary.json e simulator.json em: {input_dir}')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
simulator = json.loads(simulator_path.read_text(encoding='utf-8'))
print('MIF 2026 — auditoria reprodutível do portfólio')
print(f'Inputs usados: {input_dir}')
print(f'Caminho canônico padrão: {canonical_input_dir}')
if input_dir != canonical_input_dir:
    print('Nota de proveniência: esta execução usa uma derivação temporária dos agregados congelados; a execução autoritativa deve usar o caminho canônico.')
print(f"Transformação: {summary['meta']['transform_version']} | gerado em: {summary['meta']['generated_at']}")


MIF 2026 — auditoria reprodutível do portfólio
Inputs usados: /Users/Shared/Projects/RunnerHub/Business/_codex/analyses/mif_2026_channels/modular_dist/portfolio
Caminho canônico padrão: /Users/Shared/Projects/RunnerHub/Business/_codex/analyses/mif_2026_channels/modular_dist/portfolio
Transformação: mif-2026-portfolio.1 | gerado em: 2026-08-31T23:56:00+00:00


In [2]:
# Reconciliação do universo completo: o corte comercial não altera o total do evento.
overview = summary['overview']
overview_rows = [
    {'métrica': 'Pedidos pagos', 'valor': overview['paid_orders']},
    {'métrica': 'Inscrições pagas', 'valor': overview['paid_registrations']},
    {'métrica': 'Valor bruto (R$)', 'valor': overview['gross_value']},
    {'métrica': 'Inscrições orgânicas (referência)', 'valor': overview['organic_registrations']},
    {'métrica': 'Inscrições assistidas por cupom', 'valor': overview['coupon_assisted_registrations']},
]
show_rows(overview_rows, ['métrica', 'valor'])
assert overview['organic_registrations'] + overview['coupon_assisted_registrations'] == overview['paid_registrations']
print('Reconciliação orgânico + cupom = inscrições pagas: OK')


métrica                           | valor     
--------------------------------- | ----------
Pedidos pagos                     | 14027     
Inscrições pagas                  | 15713     
Valor bruto (R$)                  | 4321891.20
Inscrições orgânicas (referência) | 7612      
Inscrições assistidas por cupom   | 8101      
Reconciliação orgânico + cupom = inscrições pagas: OK


In [3]:
# Benchmarks por dimensão: p25 do vizinho mais próximo e p90 dos pares válidos.
benchmark_rows = []
for dimension, benchmark in summary['dimension_benchmarks'].items():
    benchmark_rows.append({
        'dimensão': dimension,
        'pares válidos': benchmark['eligible_pairs'],
        'p25 vizinho mais próximo': f"{benchmark['nearest_peer_p25_similarity_0_1']:.4f}",
        'p90 similaridade': f"{benchmark['p90_similarity_0_1']:.4f}",
    })
show_rows(benchmark_rows, ['dimensão', 'pares válidos', 'p25 vizinho mais próximo', 'p90 similaridade'])


dimensão  | pares válidos | p25 vizinho mais próximo | p90 similaridade
--------- | ------------- | ------------------------ | ----------------
geography | 2701          | 0.7105                   | 0.7124          
lot       | 2850          | 0.7710                   | 0.6772          
modality  | 2850          | 0.8965                   | 0.8682          
product   | 2850          | 0.8493                   | 0.8424          
temporal  | 2850          | 0.3396                   | 0.3254          


In [4]:
# Top 10 pares: revisão comercial descritiva, sem score mestre ou decisão automática.
redundancy_rows = []
for row in summary['redundancy_candidates'][:10]:
    similarities = ', '.join(f'{dimension}={value:.4f}' for dimension, value in sorted(row['similarities'].items()))
    redundancy_rows.append({
        'par': f"{row['left_channel']} × {row['right_channel']}",
        'dimensões': row['qualifying_dimension_count'],
        'similaridades': similarities,
        'inscrições': f"{row['left_paid_registrations']} + {row['right_paid_registrations']}",
        'bruto combinado (R$)': row['combined_gross_value'],
        'amostra': row['sample_status'],
    })
show_rows(redundancy_rows, ['par', 'dimensões', 'similaridades', 'inscrições', 'bruto combinado (R$)', 'amostra'])


par                              | dimensões | similaridades                                                                  | inscrições | bruto combinado (R$) | amostra              
-------------------------------- | --------- | ------------------------------------------------------------------------------ | ---------- | -------------------- | ---------------------
MANIADECORRIDA × ROADRUNNERS     | 5         | geography=0.8056, lot=0.8030, modality=0.9210, product=0.8976, temporal=0.7316 | 491 + 1876 | 689450.48            | referência disponível
CANALCORREDORES × ROADRUNNERS    | 5         | geography=0.7611, lot=0.7980, modality=0.9291, product=0.8617, temporal=0.5107 | 144 + 1876 | 587511.24            | referência disponível
CORRIDASFLORIPA × PACEFLORIPA    | 5         | geography=0.7554, lot=0.8552, modality=0.9105, product=0.8937, temporal=0.7045 | 372 + 444  | 227455.38            | referência disponível
CANALCORREDORES × MANIADECORRIDA | 5         | geography=0.7343, lot=0

In [5]:
# Células de dependência observada e posição dos três canais solicitados.
dependency_rows = [
    {
        'célula': f"{row['phase']} | {row['modality']} | {row['state']}",
        'canal': row['channel_name'],
        'inscrições do canal': row['paid_registrations'],
        'total evento': row['event_paid_registrations'],
        'total comercial': row['commercial_paid_registrations'],
        'exposição': row['exposure'],
    }
    for row in summary['dependency_cells'][:10]
]
show_rows(dependency_rows, ['célula', 'canal', 'inscrições do canal', 'total evento', 'total comercial', 'exposição'])
print('\nStatus no universo selecionável (> R$ 10,00 por inscrição; não orgânico):')
selectable_by_name = {row['channel_name'].casefold(): row for row in simulator['selectable_channels']}
valid_channel_types = {'assessoria', 'beneficio', 'campanha', 'comunidade', 'cortesia', 'evento_acao', 'influenciador', 'organico', 'outro', 'parceiro', 'politica'}
selectable_types = [str(row.get('channel_type', '')) for row in simulator['selectable_channels']]
assert simulator['selectable_channels']
assert all(str(row['registration_ticket']) and Decimal(str(row['registration_ticket'])) > Decimal('10.00') for row in simulator['selectable_channels'])
assert all(channel_type == channel_type.strip().casefold() and channel_type in valid_channel_types and channel_type != 'organico' for channel_type in selectable_types)
redundancy_endpoints = {endpoint.casefold() for row in summary['redundancy_candidates'] for endpoint in (row['left_channel'], row['right_channel'])}
assert redundancy_endpoints.issubset(selectable_by_name)
assert all(selectable_by_name[endpoint]['channel_type'] != 'organico' for endpoint in redundancy_endpoints)
print(f"Validação global: {len(selectable_types)} canais selecionáveis, ticket > R$ 10,00, tipo válido e não orgânico; {len(redundancy_endpoints)} endpoints de redundância não orgânicos.")
status_rows = []
for channel_name in ('ROADRUNNERS', 'Sports Week', 'PCD'):
    row = selectable_by_name.get(channel_name.casefold())
    status_rows.append({
        'canal': channel_name.upper(),
        'status': 'selecionável' if row else 'fora do universo selecionável',
        'tipo': row['channel_type'] if row else '—',
        'inscrições': row['paid_registrations'] if row else '—',
        'bruto (R$)': row['gross_value'] if row else '—',
        'ticket (R$)': row['registration_ticket'] if row else '—',
    })
show_rows(status_rows, ['canal', 'status', 'tipo', 'inscrições', 'bruto (R$)', 'ticket (R$)'])


célula                | canal       | inscrições do canal | total evento | total comercial | exposição
--------------------- | ----------- | ------------------- | ------------ | --------------- | ---------
Meio | 21K | SP       | ROADRUNNERS | 136                 | 901          | 480             | baixa    
Meio | 21K | SC       | Sports Week | 127                 | 845          | 539             | baixa    
Meio | 42K | SP       | ROADRUNNERS | 127                 | 628          | 307             | média    
Meio | 21K | SP       | Sports Week | 116                 | 901          | 480             | baixa    
Meio | 21K | SC       | ROADRUNNERS | 99                  | 845          | 539             | baixa    
Meio | 21K | PR       | Sports Week | 98                  | 681          | 343             | baixa    
Meio | 42K | SC       | Sports Week | 86                  | 540          | 321             | baixa    
Meio | 42K | SC       | ROADRUNNERS | 82                  | 540          

In [6]:
# Limites de publicação, identidade, privacidade e escopo de produto dos dois artefatos.
def nested_key_paths(value, targets, path='$'):
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f'{path}.{key}'
            if str(key).casefold() in targets:
                yield child_path
            yield from nested_key_paths(child, targets, child_path)
    elif isinstance(value, list):
        for index, child in enumerate(value):
            yield from nested_key_paths(child, targets, f'{path}[{index}]')

def scalar_value_paths(value, needle, path='$'):
    if isinstance(value, dict):
        for key, child in value.items():
            yield from scalar_value_paths(child, needle, f'{path}.{key}')
    elif isinstance(value, list):
        for index, child in enumerate(value):
            yield from scalar_value_paths(child, needle, f'{path}[{index}]')
    elif isinstance(value, str) and needle.casefold() in value.casefold():
        yield path

summary_bytes = summary_path.stat().st_size
simulator_bytes = simulator_path.stat().st_size
pii_keys = {
    'nome', 'email', 'telefone', 'celular', 'cpf', 'cnpj',
    'endereco', 'logradouro', 'numero_pedido', 'numero_inscricao',
    'data_nascimento', 'documento',
}
allowed_summary_city_path = '$.summary.overview.source_field_coverage.registrations.city'
allowed_kit_scalar_path = '$.summary.definitions.product_scope'
def assert_private_portfolio_payloads(summary_payload, simulator_payload):
    payloads = {'summary': summary_payload, 'simulator': simulator_payload}
    pii_key_paths = list(nested_key_paths(payloads, pii_keys))
    kit_key_paths = list(nested_key_paths(payloads, {'kit_incluso'}))
    kit_scalar_paths = list(scalar_value_paths(payloads, 'kit_incluso'))
    summary_city_paths = list(nested_key_paths(summary_payload, {'city'}, '$.summary'))
    simulator_city_paths = list(nested_key_paths(simulator_payload, {'city'}, '$.simulator'))
    week_start_paths = list(nested_key_paths(payloads, {'week_start'}))
    assert not pii_key_paths, pii_key_paths
    assert not kit_key_paths, kit_key_paths
    assert kit_scalar_paths == [allowed_kit_scalar_path], kit_scalar_paths
    assert summary_payload['definitions']['product_scope'] == 'Apenas a classificação kit_incluso é excluída do perfil de produto.'
    assert summary_city_paths == [allowed_summary_city_path], summary_city_paths
    assert not simulator_city_paths, simulator_city_paths
    assert not week_start_paths, week_start_paths

def assert_rejected(label, summary_payload, simulator_payload):
    try:
        assert_private_portfolio_payloads(summary_payload, simulator_payload)
    except AssertionError:
        return
    raise AssertionError(f'cópia maliciosa não rejeitada: {label}')

assert_private_portfolio_payloads(summary, simulator)
for label, target, key in (
    ('kit key no resumo', 'summary', 'kit_incluso'),
    ('city fora da metadata no resumo', 'summary', 'city'),
    ('week_start no resumo', 'summary', 'week_start'),
    ('city no simulador', 'simulator', 'city'),
    ('week_start no simulador', 'simulator', 'week_start'),
):
    malicious_summary = json.loads(json.dumps(summary))
    malicious_simulator = json.loads(json.dumps(simulator))
    (malicious_summary if target == 'summary' else malicious_simulator)[key] = False
    assert_rejected(label, malicious_summary, malicious_simulator)
assert overview['paid_orders'] == 14027
assert overview['paid_registrations'] == 15713
assert overview['gross_value'] == '4321891.20'
assert summary_bytes < 750_000
assert simulator_bytes < 2_000_000
assert simulator['dimensions'] == ['phase', 'modality', 'state', 'channel_name']
show_rows([
    {'artefato': 'summary.json', 'bytes': summary_bytes, 'limite': 750_000},
    {'artefato': 'simulator.json', 'bytes': simulator_bytes, 'limite': 2_000_000},
], ['artefato', 'bytes', 'limite'])
print('Asserções finais aprovadas: totais, limites, dimensões, PII, chaves kit_incluso e week_start ausentes; city só na metadata permitida do resumo.')
print('kit_incluso não aparece em valores escalares de saída; sua única menção é a definição que documenta esta exclusão específica.')
print('Cópias maliciosas rejeitadas: kit key, city/week_start no resumo e city/week_start no simulador.')


artefato       | bytes  | limite 
-------------- | ------ | -------
summary.json   | 221772 | 750000 
simulator.json | 819887 | 2000000
Asserções finais aprovadas: totais, limites, dimensões, PII, chaves kit_incluso e week_start ausentes; city só na metadata permitida do resumo.
kit_incluso não aparece em valores escalares de saída; sua única menção é a definição que documenta esta exclusão específica.
Cópias maliciosas rejeitadas: kit key, city/week_start no resumo e city/week_start no simulador.
